# 🏆 Week 2 Project Workbook — RAG SmartBot v1.0

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 2 · Module 5: Mini-Project (GRADED) — Add RAG to your Week 1 chatbot using product catalogs and inventory documents**

Rename this file **`Week2_Project_YourName.ipynb`** before submitting.

---

## Your mission
SmartMart's assistant must answer questions about the **full catalog and policy handbook** — accurately, with citations, refusing honestly when the knowledge base is silent — and ship **with an evaluation report** proving the quality.

## The build plan (40 min)
| Milestone | Time | What |
|---|---|---|
| M1 | 10 min | Index the knowledge: chunk the handbook (TODO), index into Chroma (Part B) |
| M2 | 10 min | Wire the bot: hybrid retriever + your grounding prompt (Part C) |
| M3 | 10 min | Pass the 10 acceptance checks (Part D) |
| M4 | 10 min | Eval report + diagnosis + limitations, submit (Parts E–G) |

## Grading — 100 points (+10 bonus)
| Category | Points |
|---|---|
| Acceptance checks (10 × 4) | 40 |
| Eval report (gold set 20+: 10 · retrieval table: 7 · RAGAS table: 8) | 25 |
| Diagnosis & fix (before/after numbers) | 10 |
| Prompt & chunking quality | 10 |
| Engineering (metadata, hybrid, sensible k/rrf_k) | 10 |
| Hygiene & honesty (no key, clean run, limitations paragraph) | 5 |
| Stretch goals (Part H) | up to +10 |

> ⚠️ **Before submitting:** `Restart & Run All` must complete cleanly with all outputs visible. Hard-coded API key = −5.

---
# Part A — Setup (do not modify)

⏳ `chromadb` takes ~1 minute — run this cell FIRST, read ahead while it installs.

In [ ]:
%pip install -q -U openai rank_bm25 chromadb

In [ ]:
import json, time, random, re
from getpass import getpass
from openai import OpenAI
import numpy as np, chromadb
from rank_bm25 import BM25Okapi

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"

def ask(prompt, temperature=0.3, max_tokens=300):
    r = client.chat.completions.create(model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content.strip()

def embed_batch(texts):
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print("✅ Setup complete.")

In [ ]:
# The catalog — 20 products with structured metadata (provided)
CATALOG = [
    ("P-101", "Prima Electric Kettle 1.5L | Rs. 1,299 | 1500W fast boil, auto shut-off, 1-year warranty | in stock: 42", "kitchen", 1299),
    ("P-102", "Prima Electric Kettle 2.0L | Rs. 1,699 | family size, keep-warm mode, 1-year warranty | in stock: 18", "kitchen", 1699),
    ("P-103", "Zenith Steel Kettle 1.0L | Rs. 899 | budget stovetop kettle, no electricity needed | in stock: 65", "kitchen", 899),
    ("P-104", "Zenith Pro Kettle 1.8L | Rs. 2,499 | premium, temperature control, 2-year warranty | in stock: 7", "kitchen", 2499),
    ("P-105", "SwiftMix Mixer Grinder 750W | Rs. 3,499 | 3 jars, overload protection, 2-year warranty | in stock: 23", "kitchen", 3499),
    ("P-106", "SwiftMix Mixer Grinder 500W | Rs. 2,299 | 2 jars, compact, 1-year warranty | in stock: 31", "kitchen", 2299),
    ("P-107", "AirChef Air Fryer 4L | Rs. 5,999 | oil-free frying, digital timer, 1-year warranty | in stock: 12", "kitchen", 5999),
    ("P-108", "AirChef Air Fryer 6L XL | Rs. 8,499 | family size, 8 presets, 2-year warranty | in stock: 4", "kitchen", 8499),
    ("P-109", "CoolBreeze Table Fan 400mm | Rs. 1,899 | 3-speed, oscillating, 2-year warranty | in stock: 55", "appliances", 1899),
    ("P-110", "CoolBreeze Tower Fan | Rs. 4,299 | remote control, timer, quiet mode | in stock: 9", "appliances", 4299),
    ("P-111", "BrightHome LED Bulb 9W (pack of 4) | Rs. 499 | warm white, 2-year replacement | in stock: 210", "electrical", 499),
    ("P-112", "BrightHome Smart Bulb WiFi | Rs. 799 | app control, 16M colors, voice assistant support | in stock: 48", "electrical", 799),
    ("P-113", "PowerSafe Extension Board 6-socket | Rs. 649 | surge protection, 2m cord | in stock: 87", "electrical", 649),
    ("P-114", "SmartWatch Fit Pro | Rs. 2,999 | heart rate, SpO2, 7-day battery | in stock: 26", "gadgets", 2999),
    ("P-115", "EarBuds Bass+ TWS | Rs. 1,499 | 24h playback, IPX4, ENC mic | in stock: 39", "gadgets", 1499),
    ("P-116", "PixelSnap Barcode Scanner X-500 | Rs. 4,999 | wireless, 18h battery, 1-year warranty | in stock: 6", "business", 4999),
    ("P-117", "ThermoSteel Water Bottle 1L | Rs. 749 | 12h hot / 24h cold, leak-proof | in stock: 120", "home", 749),
    ("P-118", "CleanSweep Robot Vacuum | Rs. 12,999 | app control, auto-dock, HEPA filter | in stock: 3", "home", 12999),
    ("P-119", "FreshBrew Coffee Maker 600ml | Rs. 2,799 | drip brew, keep-warm plate, 1-year warranty | in stock: 14", "kitchen", 2799),
    ("P-120", "ChopMaster Vegetable Chopper | Rs. 599 | 900ml, 3 blades, dishwasher safe | in stock: 73", "kitchen", 599),
]

# The messy policy handbook — Day 2's document (provided; you chunk it in Part B)
POLICY_DOC = """SmartMart Customer Service Handbook

Section 1: Returns and Refunds
Products can be returned within 7 days of purchase for a full refund, provided the customer presents the original receipt. If the receipt is lost, we offer store credit equal to the current selling price. Electronics must be unopened with seals intact. It must also be in the original packaging with all accessories included. Kitchen appliances that have been used can only be returned if defective. Defective items are eligible for free replacement within 15 days instead of refund if the customer prefers.

Section 2: Warranty Claims
All warranty claims require the original invoice and the product serial number. The standard process takes 7-10 working days. Premium members get pickup and drop service for warranty repairs at no extra charge. For products above Rs. 5,000, we offer doorstep inspection before pickup. The warranty does not cover physical damage, water damage, or unauthorized repairs. It becomes void if the product serial number is tampered with.

Section 3: EMI and Payment Options
Purchases above Rs. 3,000 are eligible for EMI on all major credit cards. We support 3, 6, 9 and 12 month tenures. No-cost EMI is available on select kitchen appliances during festival sales. UPI, cards, cash and store gift vouchers are accepted at all counters. Gift vouchers cannot be combined with no-cost EMI offers.

Section 4: SmartMart Premium Membership
Premium membership costs Rs. 499 per year. Benefits include free delivery on ALL orders regardless of value, early access to festival sales, an extra 5% discount on Tuesdays, and free pickup for warranty repairs. Membership can be cancelled within 30 days for a full refund if no benefits have been used.

Section 5: Delivery
Free delivery above Rs. 999, otherwise Rs. 49. Standard delivery takes 2-4 days in Pune and 4-7 days in the rest of Maharashtra. Same-day delivery is available in Pimple Saudagar for orders placed before 12 PM at Rs. 99. Current offer: 10% off all kitchen appliances till Sunday, not combinable with other coupons."""

print(f"Catalog: {len(CATALOG)} products | Handbook: {len(POLICY_DOC.split())} words, 5 sections")

---
# Part B — Index the knowledge (Milestone 1) 🗄️

**✏️ TODO 1: chunk the handbook context-aware** (Day 2's four rules: split on structure · one topic per chunk · **inject the section header** · overlap). A skeleton is provided — complete it.

In [ ]:
def chunk_policy_doc(doc):
    """✏️ TODO 1: context-aware chunking.
    Requirements:
      - split on 'Section N:' boundaries (regex provided below)
      - group ~3 sentences per chunk with 1 sentence overlap
      - EVERY chunk text must start with '[Handbook > <section header>] '
    Return: list of chunk strings."""
    title = doc.split("\n")[0].strip()
    sections = re.split(r"\n(?=Section \d+:)", doc)
    chunks = []
    for sec in sections[1:]:
        lines = sec.strip().split("\n")
        header = lines[0].strip()
        body = " ".join(lines[1:]).strip()
        sentences = re.split(r"(?<=[.!?]) +", body)
        # ✏️ TODO: group sentences into overlapping chunks and prepend the header tag.
        # Replace `pass` with your loop (hint: Day 2 practical, Experiment 4, strategy C)
        pass
    return chunks

POLICY_CHUNKS = chunk_policy_doc(POLICY_DOC)
assert POLICY_CHUNKS and all(c.startswith("[SmartMart Customer Service Handbook >") or c.startswith("[Handbook >") for c in POLICY_CHUNKS), \
    "TODO 1 incomplete: chunks must exist and carry the injected header"
print(f"✅ {len(POLICY_CHUNKS)} policy chunks. Sample:\n{POLICY_CHUNKS[0][:130]}...")

In [ ]:
# Index EVERYTHING into ChromaDB (provided — read it, you'll defend it in the demo)
IDS   = [r[0] for r in CATALOG] + [f"H-{i+1:02d}" for i in range(len(POLICY_CHUNKS))]
DOCS  = [f"{r[0]} | {r[1]}" for r in CATALOG] + POLICY_CHUNKS
METAS = ([{"category": r[2], "price": r[3]} for r in CATALOG]
         + [{"category": "policy", "price": -1}] * len(POLICY_CHUNKS))

chroma = chromadb.Client()
try: chroma.delete_collection("smartmart_v1")
except Exception: pass
col = chroma.create_collection("smartmart_v1", metadata={"hnsw:space": "cosine"})
col.add(ids=IDS, documents=DOCS, embeddings=embed_batch(DOCS).tolist(), metadatas=METAS)
bm25 = BM25Okapi([d.lower().replace("|", " ").replace(",", " ").split() for d in DOCS])
print(f"✅ Indexed {col.count()} chunks (catalog + handbook) with metadata")

---
# Part C — Wire the bot (Milestone 2) 🔧

**✏️ TODO 2 & 3:** hybrid retrieval parameters. **✏️ TODO 4:** your grounding prompt.

In [ ]:
def retrieve_hybrid(query, k=None, pool=None, rrf_k=60, where=None):
    """Hybrid RRF retrieval (Day 3). ✏️ TODO 2: choose k. ✏️ TODO 3: choose pool."""
    k    = k    if k    is not None else None   # ✏️ TODO 2: how many chunks reach the prompt? (hint: check 5 needs 2 facts; check 7/10 punish junk)
    pool = pool if pool is not None else None   # ✏️ TODO 3: how deep does each engine rank before fusion? (hint: Day 3 homework sweep)
    assert k and pool, "TODO 2/3 incomplete: set k and pool"
    if where:
        res = col.query(query_embeddings=[embed_batch([query])[0].tolist()], n_results=k, where=where)
        return res["documents"][0]
    scores_bm = bm25.get_scores(query.lower().replace("|", " ").replace(",", " ").split())
    bm_docs = [DOCS[i] for i in sorted(range(len(DOCS)), key=lambda i: scores_bm[i], reverse=True)[:pool]]
    res = col.query(query_embeddings=[embed_batch([query])[0].tolist()], n_results=pool)
    vec_docs = res["documents"][0]
    fused = {}
    for ranking in (bm_docs, vec_docs):
        for rank, doc in enumerate(ranking, 1):
            fused[doc] = fused.get(doc, 0.0) + 1.0 / (rrf_k + rank)
    return [d for d, _ in sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]]

In [ ]:
# ✏️ TODO 4: your grounding prompt. Start from Day 1's template and make it yours.
# Must enforce: answer ONLY from context · exact honest-IDK line · citation of IDs/sections · max 3 sentences.
RAG_TEMPLATE = """...   # ✏️ TODO 4

CONTEXT:
{context}

CUSTOMER QUESTION: {question}"""

assert "{context}" in RAG_TEMPLATE and "{question}" in RAG_TEMPLATE and len(RAG_TEMPLATE) > 200, \
    "TODO 4 incomplete: write the full grounding prompt"

class RAGSmartBot:
    """Week 2 deliverable: RAG SmartBot v1.0."""

    def __init__(self, k=None):
        self.k = k

    def say(self, question, where=None, verbose=False):
        chunks = retrieve_hybrid(question, k=self.k, where=where)
        if verbose:
            for c in chunks: print(f"   ⤷ {c[:75]}...")
        answer = ask(RAG_TEMPLATE.format(context="\n".join(chunks), question=question), temperature=0.3)
        return answer, chunks

bot = RAGSmartBot()
a, c = bot.say("what's the cheapest kettle?")
print("✅ Smoke test:", a)

---
# Part D — The 10 acceptance checks (Milestone 3) ✅

Run it. **Green = marks.** Red → the triage table (from the slides, repeated below).

| Red check | Fix first |
|---|---|
| 3 (Hinglish) | embeddings indexed? raise pool; RRF wiring |
| 5 (multi-fact) | raise k to 4–5; watch 7/10 for side effects |
| 6 (IDK trap) | tighten 'ONLY from context' + IDK line |
| 8 (faithfulness) | temperature down; citation rule; shorter answers |
| 9 (F1) | headers injected? hybrid on? sweep k on the gold set |

In [ ]:
GOLD_SET = [
    ("price of prima 1.5L kettle",              {"P-101"}),
    ("cheapest kettle",                          {"P-103"}),
    ("robot vacuum in stock?",                   {"P-118"}),
    ("X-500 scanner warranty",                   {"P-116"}),
    ("return without receipt",                   None),   # handbook — judged by keyword below
    ("delivery charge for Rs. 500 order",        None),
    ("discount on kitchen items",                None),
    ("mixer grinder with overload protection",   {"P-105"}),
    ("affordable way to boil water",             {"P-103", "P-101"}),
    ("sasta kettle dikhao",                      {"P-103"}),
    ("something to make coffee in the morning",  {"P-119"}),
    ("gift idea: music on the go",               {"P-115"}),
]
# ✏️ For full marks (Part E): extend to 20+ — add Hinglish, multi-fact, traps, typos.

def chunk_id(doc):
    return doc.split("|")[0].strip().split()[0] if "|" in doc else None

def retrieval_f1(retriever, k=3):
    f1s = []
    for query, relevant in GOLD_SET:
        if relevant is None:   # handbook queries: hit if any handbook chunk retrieved
            docs = retriever(query, k=k)
            f1s.append(1.0 if any(d.startswith("[") for d in docs) else 0.0)
            continue
        ids = [chunk_id(d) for d in retriever(query, k=k)]
        hits = [i for i in ids if i in relevant]
        p, r = len(hits) / k, len(set(hits)) / len(relevant)
        f1s.append(2*p*r/(p+r) if (p+r) else 0.0)
    return sum(f1s) / len(f1s)

FAITH_PROMPT = """Split the ANSWER into atomic factual claims; for each, is it supported by CONTEXT?
Return ONLY JSON: {{"claims": [{{"claim": "...", "supported": true}}]}}
CONTEXT:\n{context}\nANSWER:\n{answer}"""

def faithfulness(answer, chunks):
    raw = ask(FAITH_PROMPT.format(context="\n".join(chunks), answer=answer), temperature=0.0)
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    claims = json.loads(raw)["claims"]
    return sum(c["supported"] for c in claims) / max(len(claims), 1)

def n_sentences(t):
    return max(1, sum(t.count(x) for x in ".!?") - t.count("..."))

def run_acceptance_checks():
    fresh = RAGSmartBot()
    results = []

    a1, _ = fresh.say("What is the price of the Prima 1.5L kettle?")
    results.append(("C1 catalog price + citation", "1,299" in a1 and "P-101" in a1))

    a2, _ = fresh.say("Which is your cheapest kettle?")
    results.append(("C2 cheapest kettle", "899" in a2 or "Zenith Steel" in a2))

    a3, _ = fresh.say("sasta kettle dikhao")
    results.append(("C3 Hinglish query", "899" in a3 or "Zenith" in a3))

    a4, _ = fresh.say("Can I return my fan without a receipt?")
    results.append(("C4 policy: store credit", "store credit" in a4.lower()))

    a5, _ = fresh.say("What does the X-500 scanner cost and what do I need for a warranty claim?")
    results.append(("C5 multi-fact", "4,999" in a5 and ("invoice" in a5.lower() or "serial" in a5.lower())))

    a6, _ = fresh.say("Do you sell washing machines?")
    results.append(("C6 honest IDK", "don't have" in a6.lower() or "don't seem" in a6.lower() or "not" in a6.lower()))

    a7, _ = fresh.say("something for making tea", where={"$and": [{"category": {"$eq": "kitchen"}}, {"price": {"$lt": 1000}}]})
    results.append(("C7 filtered query", not any(x in a7 for x in ["1,299", "1,699", "2,499", "3,499"])))

    a8q = "What's the warranty on the SwiftMix 750W and how long do claims take?"
    a8, c8 = fresh.say(a8q)
    f = faithfulness(a8, c8)
    results.append((f"C8 faithfulness ≥ 0.8 (got {f:.2f})", f >= 0.8))

    f1 = retrieval_f1(lambda q, k=3: retrieve_hybrid(q, k=k))
    results.append((f"C9 gold-set F1 ≥ 0.60 (got {f1:.2f})", f1 >= 0.60))

    results.append(("C10 short + cited", all(n_sentences(a) <= 4 for a in [a1, a2, a4, a5]) and "P-101" in a1))

    print("=" * 52)
    passed = 0
    for name, ok in results:
        print(("✅" if ok else "❌"), name)
        passed += ok
    print("=" * 52)
    print(f"SCORE: {passed}/10 checks → {passed*4}/40 points")
    return passed

run_acceptance_checks()

---
# Part E — The eval report, layer 1: retrieval (Milestone 4) 📊

**✏️ TODO 5:** extend `GOLD_SET` above to **20+** (rubric: 10 pts). Then run the 3-retriever table:

In [ ]:
def retrieve_bm25_only(query, k=3):
    scores = bm25.get_scores(query.lower().replace("|", " ").replace(",", " ").split())
    return [DOCS[i] for i in sorted(range(len(DOCS)), key=lambda i: scores[i], reverse=True)[:k]]

def retrieve_vector_only(query, k=3):
    res = col.query(query_embeddings=[embed_batch([query])[0].tolist()], n_results=k)
    return res["documents"][0]

print(f"Gold set size: {len(GOLD_SET)} {'✅' if len(GOLD_SET) >= 20 else '⚠️ extend to 20+ for full marks'}")
print()
print(f"{'retriever':<12} {'mean F1 (k=3)':>14}")
for name, fn in [("BM25", retrieve_bm25_only),
                 ("Vector", retrieve_vector_only),
                 ("Hybrid", lambda q, k=3: retrieve_hybrid(q, k=k))]:
    print(f"{name:<12} {retrieval_f1(fn):>14.2f}")

# Part F — The eval report, layer 2: RAGAS table 📋

Day 4's harness, condensed (relevancy + context recall included). Runs 6 judged questions — ~60s:

In [ ]:
REV_Q = """Write exactly 3 short questions this answer directly answers.
Return ONLY JSON: {{"questions": ["...", "...", "..."]}}\nANSWER:\n{answer}"""

def answer_relevancy(question, answer):
    raw = ask(REV_Q.format(answer=answer), temperature=0.0)
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    qs = json.loads(raw)["questions"]
    vecs = embed_batch([question] + qs)
    return float(np.mean([cosine(vecs[0], v) for v in vecs[1:]]))

CTX_RECALL = """For each factual statement in GROUND TRUTH, is it found in CONTEXT?
Return ONLY JSON: {{"facts": [{{"fact": "...", "found": true}}]}}
GROUND TRUTH: {truth}\nCONTEXT:\n{context}"""

def context_recall(truth, chunks):
    raw = ask(CTX_RECALL.format(truth=truth, context="\n".join(chunks)), temperature=0.0)
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    facts = json.loads(raw)["facts"]
    return sum(f["found"] for f in facts) / max(len(facts), 1)

EVAL_SET = [
    {"q": "cheapest kettle you have?",            "truth": "The Zenith Steel Kettle 1.0L is cheapest at Rs. 899."},
    {"q": "sasta kettle dikhao",                  "truth": "The budget option is the Zenith Steel Kettle at Rs. 899."},
    {"q": "return without receipt — what do I get?", "truth": "Without a receipt you get store credit equal to the current selling price."},
    {"q": "X-500 price and warranty claim needs?",   "truth": "The X-500 costs Rs. 4,999; claims need the original invoice and serial number."},
    {"q": "what do premium members get on Tuesdays?", "truth": "Premium members get an extra 5% discount on Tuesdays."},
    {"q": "do you sell washing machines?",           "truth": "SmartMart does not carry washing machines; the bot should say it doesn't have that information."},
]

bot = RAGSmartBot()
rows = []
for item in EVAL_SET:
    answer, chunks = bot.say(item["q"])
    try: f = faithfulness(answer, chunks)
    except Exception: f = float("nan")
    try: r = answer_relevancy(item["q"], answer)
    except Exception: r = float("nan")
    try: cr = context_recall(item["truth"], chunks)
    except Exception: cr = float("nan")
    rows.append((item["q"], f, r, cr))

print(f"{'question':<42} {'faith':>6} {'relev':>6} {'ctx_R':>6}")
print("-" * 64)
for q, f, r, cr in rows:
    print(f"{q[:40]:<42} {f:>6.2f} {r:>6.2f} {cr:>6.2f}")
print("-" * 64)
print(f"{'MEAN':<42} {np.nanmean([r[1] for r in rows]):>6.2f} "
      f"{np.nanmean([r[2] for r in rows]):>6.2f} {np.nanmean([r[3] for r in rows]):>6.2f}")

---
# Part G — Diagnosis & fix (10 pts) + limitations (part of the 5) 🔬

**✏️ TODO 6:** find your worst number in Parts E/F, name its triage dial, change ONE thing, re-run, show before/after:

In [ ]:
# ✏️ TODO 6 — fill in your diagnosis (template):
DIAGNOSIS = """
WORST METRIC : (e.g. ctx_recall on 'X-500 price and warranty claim needs?' = 0.50)
TRIAGE DIAL  : (which dial → which component)
ONE FIX      : (e.g. raised k from 3 to 4 in retrieve_hybrid)
BEFORE       : (number)
AFTER        : (number — re-run the relevant cell/question below and paste)
"""
print(DIAGNOSIS)

# Re-run your fixed question here so the AFTER number is visible in the submitted notebook:
# answer, chunks = RAGSmartBot(k=4).say("...")
# print(faithfulness(answer, chunks) / context_recall(..., chunks) etc.)

**✏️ TODO 7 — Known limitations (write 4–6 honest sentences here):**

*Replace this text: what does your v1.0 still do badly? (Think: judge reliability, gold-set size, stale stock counts in chunk text, no live inventory, threshold-less retrieval on nonsense queries, Hinglish coverage…) Honesty scores points; a grader who finds a weakness you didn't list scores it against you.*

---
# Part H — Stretch goals (optional, up to +10) 🚀

**Only after all 10 checks pass.**

**+4 · Confidence guardrail** — before calling the LLM, check the top vector similarity; below a threshold you choose, answer "we don't seem to carry that" without spending tokens. Show it firing on 3 nonsense queries ("do you sell helicopters?").

**+3 · Multi-query retrieval** — LLM writes 2 paraphrases; retrieve for all 3; RRF-merge. Show one query it rescues.

**+3 · Persistence** — `chromadb.PersistentClient(path=...)` + a guard that skips re-embedding when the collection exists. Prove zero embedding calls on second run.

In [ ]:
# ✏️ Stretch goal workspace (optional)


---
# ✅ Submission checklist

1. `Runtime → Restart & Run All` — completes cleanly
2. Part D shows check results (aim 10/10 ✅) · Parts E & F tables visible
3. Part G diagnosis has real before/after numbers · limitations written
4. Renamed `Week2_Project_YourName.ipynb` · **no API key in the file**
5. Submit via the class channel before the deadline

---
# 📎 Appendix — Reference values (unlocked after submission)

<details>
<summary><b>Open ONLY after submitting</b></summary>

- **TODO 1:** Day 2 practical, Experiment 4, strategy C — verbatim `chunk_context_aware` works (loop: `for i in range(0, len(sentences), 2): group = sentences[i:i+3]; chunks.append(f"[{title} > {header}] " + " ".join(group)); if i+3 >= len(sentences): break`)
- **TODO 2:** `k = 4` passes all checks (3 also works if your prompt is tight; 5+ risks C7/C10)
- **TODO 3:** `pool = 10`
- **TODO 4:** Day 1's RAG_TEMPLATE + explicit citation instruction ("Cite the product/policy ID or [Handbook > Section] you used") + the exact IDK line
- **Expected:** F1 ≈ 0.7–0.85 hybrid, faithfulness ≈ 0.9+, C1–C10 all green

</details>

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*
**Week 2 complete. Monday: Agentic AI — your bot starts DOING.** 🎉